# 🎙️ Open-ElevenLabs Voice Studio (CosyVoice 2 SOTA)
### Премиальное нейросетевое клонирование и синтез русской речи уровня ElevenLabs
---
> * 💎 **CosyVoice 2 (0.5B)**: Qwen2.5 LLM backbone + Causal Flow Matching.
> * 🇷🇺 **Идеальный русский язык**: Нативное понимание грамматики, ударений и ритма поэзии без костылей и артефактов.
> * 📂 **Professional VoiceLab**: Обучение и клонирование голоса из любого количества аудиозаписей.
> * ⚡ **A100 Оптимизация**: Генерация целых стихотворений и страниц текста за секунды.
> * 💾 **Google Drive Sync**: Ваши обученные голоса сохраняются навсегда в .

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markankamaransheogarat-sys/open-elevenlabs-voice-studio/blob/main/Open_ElevenLabs_Voice_Studio_F5_TTS.ipynb)

## 1. ⚙️ Установка окружения (Setup)
Нажмите кнопку **Play** слева на ячейке ниже. Установка займет ~1-2 минуты.

In [ ]:
# @title ⚙️ 1. Установка окружения ElevenLabs SOTA (CosyVoice 2 + Qwen2.5)
# Нажмите кнопку Play слева (установка займет ~1-2 минуты)

import os
import sys

print("🚀 [1/4] Подключение Google Drive для вечного сохранения голосов...")
try:
    from google.colab import drive
    drive.mount('/content/drive')
    VOICES_DIR = '/content/drive/MyDrive/ElevenLabs_Voices'
    print(f"💾 Google Drive подключен! Папка голосов: {VOICES_DIR}")
except Exception:
    VOICES_DIR = '/content/voices'
    print(f"💾 Работаем локально в: {VOICES_DIR}")

os.makedirs(VOICES_DIR, exist_ok=True)

print("📦 [2/4] Установка системных аудиоутилит (SoX, FFmpeg)...")
os.system("apt-get update -qq && apt-get install -y -qq sox libsox-dev ffmpeg > /dev/null 2>&1")

print("📦 [3/4] Клонирование CosyVoice 2...")
if not os.path.exists("/content/CosyVoice"):
    os.system("git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git /content/CosyVoice")
else:
    os.system("cd /content/CosyVoice && git submodule update --init --recursive")

print("📦 [4/4] Установка нейросетевых библиотек...")
packages = [
    "HyperPyYAML",
    "conformer",
    "diffusers",
    "hydra-core",
    "inflect",
    "librosa",
    "lightning",
    "modelscope",
    "omegaconf",
    "onnx",
    "onnxruntime-gpu",
    "openai-whisper",
    "soundfile",
    "transformers>=4.40.0,<=4.45.2",
    "x-transformers",
    "wetext",
    "pydub",
    "huggingface_hub",
    "gradio",
    "regex",
    "wget",
    "rootutils",
    "Unidecode",
    "einops",
    "gdown"
]

for pkg in packages:
    os.system(f"pip install -q {pkg}")

print("⬇️ Загрузка SOTA модели CosyVoice2-0.5B (~1.8 GB)...")
model_dir = '/content/CosyVoice/pretrained_models/CosyVoice2-0.5B'
if not os.path.exists(model_dir):
    try:
        from huggingface_hub import snapshot_download
        print("Скачивание весов с Hugging Face...")
        snapshot_download(repo_id='FunAudioLLM/CosyVoice2-0.5B', local_dir=model_dir)
    except Exception as e:
        print(f"Hugging Face fallback на ModelScope: {e}")
        from modelscope import snapshot_download
        snapshot_download('iic/CosyVoice2-0.5B', local_dir=model_dir)

print("\n============================================================")
print("✨ ВСЁ ГОТОВО! Переходите к Ячейке №2 ниже для запуска Студии!")
print("============================================================")


## 2. 🎙️ Запуск Студии Озвучки и VoiceLab
Нажмите кнопку **Play** на ячейке ниже. Появится публичная ссылка  для работы с телефона и ПК.

In [ ]:
# @title 🎙️ 2. Запуск ElevenLabs Web-Studio (CosyVoice 2 + VoiceLab)
# Нажмите кнопку Play слева для открытия интерфейса

import os
import sys
import re

# Проверка и авто-установка зависимостей (Self-Healing)
required_modules = [
    ("HyperPyYAML", "hyperpyyaml"),
    ("wetext", "wetext"),
    ("modelscope", "modelscope"),
    ("conformer", "conformer"),
    ("diffusers", "diffusers"),
    ("hydra-core", "hydra"),
    ("inflect", "inflect"),
    ("librosa", "librosa"),
    ("lightning", "lightning"),
    ("omegaconf", "omegaconf"),
    ("onnx", "onnx"),
    ("onnxruntime", "onnxruntime"),
    ("openai-whisper", "whisper"),
    ("soundfile", "soundfile"),
    ("transformers>=4.40.0,<=4.45.2", "transformers"),
    ("pydub", "pydub"),
    ("gradio", "gradio"),
    ("regex", "regex"),
    ("wget", "wget"),
    ("rootutils", "rootutils"),
    ("Unidecode", "unidecode"),
    ("einops", "einops"),
    ("gdown", "gdown")
]

for pkg, mod in required_modules:
    try:
        __import__(mod)
    except ImportError:
        print(f"📦 Доустановка {pkg}...")
        os.system(f"pip install -q {pkg}")

# Защита от несовместимых версий transformers (>=4.46 или 5.x)
try:
    import transformers
    from packaging import version
    if version.parse(transformers.__version__) > version.parse("4.45.2") or version.parse(transformers.__version__) < version.parse("4.40.0"):
        print(f"📦 Корректировка версии transformers ({transformers.__version__} -> 4.45.2)...")
        os.system("pip install -q 'transformers>=4.40.0,<=4.45.2'")
except Exception:
    pass

if not os.path.exists("/content/CosyVoice"):
    print("🚀 Клонирование CosyVoice...")
    os.system("git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git /content/CosyVoice")

# Добавляем пути CosyVoice и Matcha-TTS
for p in ['/content/CosyVoice', '/content/CosyVoice/third_party/Matcha-TTS']:
    if p not in sys.path:
        sys.path.append(p)

import glob
import json
import time
import shutil
import torch
import torchaudio
import numpy as np
from pydub import AudioSegment
import whisper
import gradio as gr

# Проверка GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f"⚡ Устройство: {device} ({gpu_name})")

VOICES_DIR = '/content/drive/MyDrive/ElevenLabs_Voices' if os.path.exists('/content/drive/MyDrive') else '/content/voices'
os.makedirs(VOICES_DIR, exist_ok=True)

# Проверка наличия весов
model_dir = '/content/CosyVoice/pretrained_models/CosyVoice2-0.5B'
if not os.path.exists(model_dir):
    print("⬇️ Загрузка весов модели CosyVoice2-0.5B...")
    try:
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id='FunAudioLLM/CosyVoice2-0.5B', local_dir=model_dir)
    except Exception as e:
        print(f"Hugging Face fallback на ModelScope: {e}")
        from modelscope import snapshot_download
        snapshot_download('iic/CosyVoice2-0.5B', local_dir=model_dir)

# Инициализация CosyVoice 2 с защитным циклом Self-Healing
print("🎙️ Загрузка CosyVoice2-0.5B в память...")
cosyvoice = None
for attempt in range(10):
    try:
        from cosyvoice.cli.cosyvoice import AutoModel
        cosyvoice = AutoModel(model_dir=model_dir)
        break
    except Exception as e:
        err_str = str(e)
        m = re.search(r"No module named '([^']+)'", err_str)
        if m:
            missing_pkg = m.group(1)
            print(f"📦 [Self-Healing] Доустановка отсутствующего модуля: {missing_pkg}...")
            os.system(f"pip install -q {missing_pkg}")
        else:
            raise e

if cosyvoice is None:
    raise RuntimeError("Не удалось инициализировать CosyVoice 2 после нескольких попыток.")

sample_rate = cosyvoice.sample_rate
print(f"✅ CosyVoice 2 активна! Студийный звук: {sample_rate} Hz")

# Загрузка Whisper для мгновенной расшифровки
print("🧠 Загрузка Whisper ASR (автоматическая расшифровка)...")
whisper_model = whisper.load_model('base', device=device)
print("✅ Whisper готов!")

# Синхронизация сохраненных голосов
def sync_saved_voices():
    # Автоматическое создание демонстрационного голоса по умолчанию, если библиотека пуста
    default_voice_dir = os.path.join(VOICES_DIR, "CosyVoice Studio (Default)")
    default_wav_src = "/content/CosyVoice/asset/zero_shot_prompt.wav"
    if os.path.exists(default_wav_src) and not os.path.exists(default_voice_dir):
        try:
            os.makedirs(default_voice_dir, exist_ok=True)
            shutil.copy(default_wav_src, os.path.join(default_voice_dir, "ref.wav"))
            with open(os.path.join(default_voice_dir, "ref.txt"), "w", encoding="utf-8") as f:
                f.write("希望你以后能够做的比我还好呦。")
            print("✨ Создан эталонный голос по умолчанию: 'CosyVoice Studio (Default)'")
        except Exception as e:
            print(f"Не удалось инициализировать дефолтный голос: {e}")

    voices = []
    if os.path.exists(VOICES_DIR):
        for d in sorted(os.listdir(VOICES_DIR)):
            v_path = os.path.join(VOICES_DIR, d)
            if os.path.isdir(v_path):
                wav_path = os.path.join(v_path, "ref.wav")
                txt_path = os.path.join(v_path, "ref.txt")
                if os.path.exists(wav_path) and os.path.exists(txt_path):
                    voices.append(d)
    return voices

saved_voices = sync_saved_voices()
print(f"📚 Загружено сохраненных голосов: {len(saved_voices)}")

# Умная разбивка поэзии и длинных текстов
def split_russian_text(text, max_len=180):
    text = text.strip()
    if not text:
        return []
    
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    chunks = []
    curr_chunk = ""
    
    for line in lines:
        if len(curr_chunk) + len(line) + 1 <= max_len:
            curr_chunk = f"{curr_chunk} {line}".strip() if curr_chunk else line
        else:
            if curr_chunk:
                chunks.append(curr_chunk)
            if len(line) <= max_len:
                curr_chunk = line
            else:
                sentences = re.split(r'([.!?…;]+)', line)
                temp = ""
                for s in sentences:
                    if len(temp) + len(s) <= max_len:
                        temp += s
                    else:
                        if temp.strip():
                            chunks.append(temp.strip())
                        temp = s
                curr_chunk = temp.strip()
                
    if curr_chunk:
        chunks.append(curr_chunk)
        
    return chunks

# Синтез речи через CosyVoice 2 (Zero-Shot + Cross-Lingual Fallback)
def synthesize_speech(voice_name, text, speed):
    if not voice_name:
        return None, "⚠️ Пожалуйста, выберите голос из списка или создайте новый во вкладке VoiceLab."
    if not text.strip():
        return None, "⚠️ Введите текст для озвучивания."

    wav_path = os.path.join(VOICES_DIR, voice_name, "ref.wav")
    txt_path = os.path.join(VOICES_DIR, voice_name, "ref.txt")
    if not os.path.exists(wav_path):
        return None, f"⚠️ Файл эталона голоса '{voice_name}' не найден в {VOICES_DIR}."

    prompt_text = ""
    if os.path.exists(txt_path):
        with open(txt_path, "r", encoding="utf-8") as f:
            prompt_text = f.read().strip()

    start_time = time.time()
    chunks = split_russian_text(text, max_len=180)
    if not chunks:
        return None, "⚠️ Текст не содержит слов для озвучивания."

    print(f"🎙️ Синтез: голос='{voice_name}' | фрагментов={len(chunks)} | скорость={speed}x")

    audio_segments = []
    pause_samples = int(sample_rate * 0.12)
    silence_tensor = torch.zeros(1, pause_samples)
    last_err = None

    for idx, chunk in enumerate(chunks):
        if not chunk.strip():
            continue

        chunk_success = False

        # Попытка 1: Zero-shot с эталонным текстом (если prompt_text доступен)
        if prompt_text:
            try:
                for out in cosyvoice.inference_zero_shot(
                    chunk,
                    prompt_text,
                    wav_path,
                    stream=False,
                    speed=float(speed),
                    text_frontend=False
                ):
                    wav = out['tts_speech'].cpu()
                    if wav.shape[1] > 0:
                        audio_segments.append(wav)
                        chunk_success = True
            except Exception as e_zero:
                print(f"⚠️ Zero-shot предупреждение для фрагмента '{chunk[:30]}...': {e_zero}")

        # Попытка 2: Гарантированный Cross-Lingual режим клонирования тембра (Flow Matching)
        if not chunk_success:
            try:
                for out in cosyvoice.inference_cross_lingual(
                    chunk,
                    wav_path,
                    stream=False,
                    speed=float(speed),
                    text_frontend=False
                ):
                    wav = out['tts_speech'].cpu()
                    if wav.shape[1] > 0:
                        audio_segments.append(wav)
                        chunk_success = True
            except Exception as e_cross:
                last_err = e_cross
                import traceback
                traceback.print_exc()
                print(f"❌ Ошибка в cross-lingual: {e_cross}")

        if chunk_success and idx < len(chunks) - 1:
            audio_segments.append(silence_tensor)

    if not audio_segments:
        err_msg = f": {last_err}" if last_err else ""
        return None, f"❌ Ошибка генерации аудио{err_msg}. Проверьте консоль Colab."

    full_wav = torch.cat(audio_segments, dim=1)
    audio_np = (full_wav.squeeze(0).numpy() * 32767).clip(-32768, 32767).astype(np.int16)
    
    elapsed = time.time() - start_time
    duration = len(audio_np) / sample_rate
    info = f"✅ Готово за {elapsed:.1f} сек! (Длительность аудио: {duration:.1f} с, RTF: {elapsed/duration:.2f}x)"
    
    return (sample_rate, audio_np), info

# Создание и обучение нового голоса через VoiceLab
def create_voice(voice_name, audio_files):
    if not voice_name or not voice_name.strip():
        return gr.update(), None, "⚠️ Введите название голоса (например: 'Есенин')."
    if not audio_files:
        return gr.update(), None, "⚠️ Загрузите хотя бы один аудиофайл."

    voice_name = re.sub(r'[^a-zA-Z0-9а-яА-ЯёЁ_\- ]', '', voice_name).strip()
    if not voice_name:
        return gr.update(), None, "⚠️ Введите корректное название голоса."

    target_dir = os.path.join(VOICES_DIR, voice_name)
    os.makedirs(target_dir, exist_ok=True)

    print(f"📂 Обучение голоса '{voice_name}' из {len(audio_files)} файлов...")

    combined = AudioSegment.empty()
    for f in audio_files:
        try:
            filepath = f.name if hasattr(f, 'name') else f
            seg = AudioSegment.from_file(filepath)
            seg = seg.set_channels(1).set_frame_rate(16000)
            combined += seg
        except Exception as e:
            print(f"Предупреждение при чтении {f}: {e}")

    if len(combined) < 1500:
        return gr.update(), None, "❌ Аудио слишком короткое. Загрузите больше записей."

    combined = combined.normalize()

    target_wav_path = os.path.join(target_dir, "ref.wav")
    # Оптимальная длина эталона для CosyVoice: 5-8 секунд
    if len(combined) > 10000:
        start_ms = max(0, int((len(combined) - 8000) / 2))
        ref_segment = combined[start_ms:start_ms + 8000]
    else:
        ref_segment = combined

    ref_segment.export(target_wav_path, format="wav")

    # Whisper транскрибация с автоопределением языка
    print("📝 Whisper расшифровывает эталонный сэмпл...")
    try:
        res = whisper_model.transcribe(target_wav_path)
        prompt_text = res['text'].strip()
    except Exception as e:
        print(f"Ошибка Whisper транскрибации: {e}")
        prompt_text = ""

    target_txt_path = os.path.join(target_dir, "ref.txt")
    with open(target_txt_path, "w", encoding="utf-8") as f:
        f.write(prompt_text)

    voices = sync_saved_voices()
    disp_text = prompt_text if prompt_text else "Режим прямого клонирования тембра (Cross-Lingual)"
    info = f"🎉 Голос '{voice_name}' успешно обучен и сохранен!\n📝 Распознанный текст: «{disp_text}»\n💾 Сохранено навсегда на Google Drive!"

    return gr.update(choices=voices, value=voice_name), target_wav_path, info

def load_voice_preview(voice_name):
    if not voice_name:
        return None
    wav_path = os.path.join(VOICES_DIR, voice_name, "ref.wav")
    return wav_path if os.path.exists(wav_path) else None

# Построение интерфейса Gradio
theme = gr.themes.Soft(primary_hue="amber", neutral_hue="slate")
default_poem = '''Не жалею, не зову, не плачу,
Всё пройдет, как с белых яблонь дым.
Увяданья золотом охваченный,
Я не буду больше молодым.

Ты теперь не так уж будешь биться,
Сердце, тронутое холодком,
И страна березового ситца
Не заманит шляться босиком.'''

with gr.Blocks(theme=theme, title="🎙️ ElevenLabs SOTA Voice Studio") as demo:
    gr.Markdown("# 🎙️ ElevenLabs Professional Voice Studio (CosyVoice 2 SOTA)")
    gr.Markdown("*Премиальное нейросетевое клонирование голоса и генерация русской речи на базе Qwen2.5 LLM + Causal Flow Matching.*")

    with gr.Tabs():
        with gr.Tab("🎙️ Студия озвучки (Studio)"):
            with gr.Row():
                with gr.Column(scale=1):
                    voice_dropdown = gr.Dropdown(
                        choices=saved_voices,
                        value=saved_voices[0] if saved_voices else None,
                        label="Выберите обученный голос",
                        interactive=True
                    )
                    preview_audio = gr.Audio(
                        value=load_voice_preview(saved_voices[0]) if saved_voices else None,
                        label="🎵 Как звучит эталон голоса",
                        interactive=False
                    )
                    speed_slider = gr.Slider(minimum=0.6, maximum=1.5, value=1.0, step=0.05, label="Скорость речи (Speed)")
                    refresh_btn = gr.Button("🔄 Обновить список голосов")

                with gr.Column(scale=2):
                    text_input = gr.Textbox(
                        lines=10,
                        value=default_poem,
                        label="Введите любой русский текст или стихи",
                        placeholder="Вставьте стихи, диалог или статью..."
                    )
                    synth_btn = gr.Button("⚡ Озвучить выбранным голосом (CosyVoice 2)", variant="primary", size="lg")
                    synth_status = gr.Markdown("")
                    output_audio = gr.Audio(label="🎧 Результат (Студийный WAV 24kHz)", interactive=False)

        with gr.Tab("🔬 VoiceLab (Обучение нового голоса)"):
            gr.Markdown("### 📂 Загрузите аудиозаписи для обучения нового голоса")
            gr.Markdown("> Вы можете загрузить 1, 3, 5 или 10 аудиозаписей (голосовые заметки, диктофон, отрывки видео). Нейросеть автоматически очистит их, сделает конкатенацию, распознает текст через Whisper и сохранит голос навсегда в Google Drive.")
            with gr.Row():
                with gr.Column():
                    new_voice_name = gr.Textbox(label="Название нового голоса (например, 'Есенин' или 'Мой Голос')")
                    audio_upload = gr.File(
                        file_count="multiple",
                        file_types=["audio"],
                        label="Загрузите аудиозаписи (WAV, MP3, M4A, OGG)"
                    )
                    train_btn = gr.Button("🔥 Обучить и сохранить голос в библиотеку", variant="primary")
                with gr.Column():
                    train_status = gr.Markdown("")
                    train_preview = gr.Audio(label="Очищенный эталонный сэмпл голоса", interactive=False)

    voice_dropdown.change(load_voice_preview, inputs=[voice_dropdown], outputs=[preview_audio])
    refresh_btn.click(lambda: gr.update(choices=sync_saved_voices()), outputs=[voice_dropdown])

    synth_btn.click(
        synthesize_speech,
        inputs=[voice_dropdown, text_input, speed_slider],
        outputs=[output_audio, synth_status]
    )

    train_btn.click(
        create_voice,
        inputs=[new_voice_name, audio_upload],
        outputs=[voice_dropdown, train_preview, train_status]
    )

print("🚀 Запуск публичной ссылки...")
demo.queue().launch(share=True, debug=False)
